In [3]:
vault = "https://kv-itra-fioda.vault.azure.net/"
key = notebookutils.credentials.getSecret(vault, "openaq-api-key")

import json
import requests
import pandas as pd
from datetime import datetime
import time

def fetch_sensors_days(sensor_id):
    all_results = []
    page = 1
    date = datetime.now()
    while True:
        for attempt in range(3):
            response = requests.get(
                f"https://api.openaq.org/v3/sensors/{sensor_id}/days",
                headers={"X-API-Key": key},
                params={"date_from": "2016-07-01", "date_to": datetime.strftime(date, '%Y-%m-%d'), "limit": 1000, "page": page},
            )
            if response.status_code == 200:
                break
            time.sleep(2 * (attempt + 1))
        else:
            response.raise_for_status()
        results = response.json()['results']
        all_results.extend(results)

        if len(results) < 1000:
            break
        page += 1
        time.sleep(1)
    return all_results

sensors = pd.read_csv("/lakehouse/default/Files/reference/openaq_sensors.csv")
sensors = sensors[sensors['parameter'].isin(['pm25','no2'])]

for _, row in sensors.iterrows():
    sensor_id = row['sensor_id']
    res = fetch_sensors_days(sensor_id)
    with open(f"/lakehouse/default/Files/air/openaq_days_{sensor_id}.json", "w", encoding="utf-8") as f:
        json.dump(res, f, ensure_ascii=False)

    values = [r["value"] for r in res if r["value"] is not None]
    print(
        sensor_id, row['parameter'], len(res),
        res[0]["period"]["datetimeFrom"]["local"][:10],
        res[-1]["period"]["datetimeFrom"]["local"][:10],
        "low_cov:", sum(1 for r in res if r["coverage"]["percentCoverage"] < 75),
        "none:", sum(1 for r in res if r["value"] is None),
        "neg:", sum(1 for v in values if v < 0),
    )
    time.sleep(1)

StatementMeta(, 82019324-0f87-4881-aff1-667c1a3f652d, 10, Finished, Available, Finished, False)

673 pm25 3237 2016-08-04 2026-11-19 low_cov: 820 none: 3 neg: 0
1097 pm25 3163 2016-08-03 2026-11-07 low_cov: 715 none: 2 neg: 2
1102 pm25 2367 2016-08-03 2026-11-19 low_cov: 658 none: 2 neg: 0
1103 pm25 3253 2016-08-03 2026-11-19 low_cov: 773 none: 6 neg: 2
1152 pm25 2911 2016-08-03 2026-11-19 low_cov: 625 none: 1 neg: 0
1128 pm25 3371 2016-08-03 2026-11-19 low_cov: 794 none: 2 neg: 0
1145 pm25 2829 2016-08-03 2026-11-19 low_cov: 714 none: 5 neg: 0
1146 pm25 2357 2016-08-03 2026-11-19 low_cov: 467 none: 5 neg: 0
1535 no2 3577 2016-07-01 2026-11-19 low_cov: 618 none: 0 neg: 0
1534 pm25 3451 2016-07-01 2026-11-19 low_cov: 646 none: 2 neg: 0
23341 no2 2988 2017-07-13 2026-11-19 low_cov: 515 none: 0 neg: 0
2644 no2 3607 2016-07-01 2026-11-19 low_cov: 605 none: 0 neg: 0
